## Automagically making fit molvis commands for fitting alphafold-predicted model and experimentally determined protein structure of part of the protein using PDBsum data to align Nterminal numbering

(Note **you may be better off just using PyMOL's `super` command**, like so `super mobile_object, target_object`, because that includes automatically finding structurally equivalent regions and  works great even when structures share only partial overlap so that it handles cases where one structure is a fragement of another. **I need to explore this more**.... However, I imagine running this notebook as a guide would help you consider if PyMOL did things right and probably still be recommended if you need to double-check something. If you need very precise control over the specific residue pairs, you'll want this notebook, or related ones, for sure.)

This notebook is part of my series to help guide and assess and work out residue numbering equivlaency between models for fit commands. This one specifically targets when you have an experimental structure and a AlphaFold model that is basically the same protein. (Minor allele and strain differences should not matter much as long as there aren't insertions/deletions. Even if there are insertions or deletions, **the process of running this notebook could still offer you substantial guidance on what to consider for adapting any offered fit commands to be accurate**, see the 'MAJOR CAVEATS TO KEEP IN MIND' section below for more informaton.) The other notebooks in this series (I'm not sure if this is exhaustive at this time but following related notes/breadcrumbs should help find any others):
- `Determine residues that match to a reference from MSA and use to construct fit commands.ipynb` (in my cl_demo-binder repo)
- `Using Biopython PDB module to list resolved residues and construct fit commands.ipynb` (in my cl_demo-binder repo)
- `GSD Conversion of FASTA alignment format to clustal and making of molvis fit commands.ipynb` (in my cl_demo-binder repo)
(Reminder for myself, I can find associated documents & specific cases of runs to support research in my local stuff by searching 'fit commands' in title and follow file breadcrumbs elsewhere.)

Alphafold is trained on previous experimental data and more than happy to greedily predict structure for any sequence submit you, filling in any segment with unknown experimentally-determined structure with some predicted form.  
This penchant to fill in segments where experimental data is missing, however, can then make superposition or 'fitting' the determined and predicted structures together difficult, especially if you want it very accurate.

This notebook will generate the fit commands for PyMOL and Jmol for a structure and the corresponding AlphaFold of the entire protein or contiguous domain/segment.

It builds on some of the things I worked out in the notebooks `Using Biopython PDB module to list resolved residues and construct fit commands.ipynb` and `Determine residues that match to a reference from MSA and use to construct fit commands.ipynb`, that are located along side this notebook.   
The process relies on:
- the PDB file includes information on missing residues that can be parsed
- the PDBsum site shares the FASTA of only the residues resolved in the structure, and so the sequence submitted to AlphaFold (FASTA format) can be used to adjust the numbering so that at least the  by the script including a step where the submitted sequence is aligned with the sequence resolved in the structure to determine the offset at the start.

MAJOR CAVEATS TO KEEP IN MIND: 
- the code in this notebook assumes the segments involved are contiguous and that any sequence differences, whether due to allele, strain, construct, differences, etc.,  between the experimental model and the prediction don't disrupt that continuity. Hence, internal insertions/deletions would mean you could not just blindly trust the 'automagically' produced fit commands. However, as part of the process, an alignment is produced that would probably be a great guide in adapting the offered commands, and so I would still suggest running this notebook to help you work things out.
- structure biologists do a lot of weird stuff to get at the strucutural data, such as making chimeras and numbering things stangeling and so don't be surprised if the script breaks. YMMV. It may just be a matter of editing the input yourself some. (I may need to make overrides possible?!?! Such as providing a way to set things beyond the N-terminal correction value, etc.? **<---Possible 'to-do' options for later**)

An example follows. It is meant to be adaptable to use the PDB codes of structures that interest you.  
Before editing the code to customize it to the data of interest to you, you may wish to work through the demonstration first so you know what to expect.

Information about example used as demonstration:
- [PDB id code: 1nql](https://www.rcsb.org/structure/1NQL) , chain A, which is the epidermal growth factor receptor (EGFR)
- [PDBsum record sequence of the resolved protein for PDB record 1nql](https://www.ebi.ac.uk/thornton-srv/databases/cgi-bin/pdbsum/GetText.pl?pdb=1nql&chain=A&seq_fasta=1)

----

### Preparation

A number of values need to be set and some supporting data obtained before running the main part of this script.

The next cell is used to define the structures of interest. The PDB id code identifier needs to be supplied, and the specific chain for the protein you wish to match up with the AlphaFold result of either the entire protein or at least a continuous segment.

In [1]:
structure = "1nql"
chain = "A"

Next you need to assign submitted_seq as the sequence of the AlphaFold input protein sequence.  
(This sequence should be what you submitted to AlphaFold and is usually listed along with the `inputs` among the results if you need to recover it. You don't want to include the description line. Just the main sequence of amino acids.) (DEVELOPMENT NOTE: LOOK into seeing if line breaks are going to be a problem? **For now assuming they are removed by user.**)

In [2]:
submitted_seq = "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPS"

For some reason, even though you provide chain designation in call to PDBsum, you also need to provide the number at the end of the URL. This corresponds to the number it is as listed of the chains of the structure at the PDBsum. In most cases, if your chain is the first one, or second one, the number will be `1` or `2`, respectively. Since some have much different ordering (Example: [6agb chain B](https://www.ebi.ac.uk/thornton-srv/databases/cgi-bin/pdbsum/GetPage.pl?pdbcode=6agb&template=protein.html&r=wiring&l=1&chain=B) is first and so the number is `1`), it isn't easy to automate. The best way is to go to the page with the FASTA sequence and copy the last number into the cell below  (In relation to the example, for chain J of [the example](https://www.ebi.ac.uk/thornton-srv/databases/cgi-bin/pdbsum/GetText.pl?pdb=6agb&chain=J&seq_fasta=9) the number would need to be assigned be 9 here from `https://www.ebi.ac.uk/thornton-srv/databases/cgi-bin/pdbsum/GetText.pl?pdb=6agb&chain=J&seq_fasta=9`.):

In [3]:
pdbsum_num = 1

Those four pieces of information should be all you need to enter.  

Note, in the process of running this notebook, the code will include determining the residue number designated for the first residue observed in the experimentally determined structure. It will report this as `first_residue_numberEXP` and use it to align the start of the numbering of the two sequences.

In most cases you'd be looking at the AlphaFold-produced model prediciton the first residue would be expected be numbered `1`. Or treated as the equivalent of that number if you are looking at a domain or other portion of the sequence and the experimentalist used number relative that.  
Though expected to be rare, I can imagine some case coming up where the numbering in the AlphaFold-produced model would not be the number `1`, for example, maybe if the AlphaFold-predicted structure had been renumbered to match some other convention. So you could potentially (see development note!!-->) adjust things with this next setting. (DEVELOPMENT NOTE: This is not yet implemented. JUST MAKING A PLACEHOLDER NOTE FOR NOW.)

In [4]:
# Presumably the first residue of the AlphaFold model will be residue number 1 but that may not be the actual case and you'd adjust that here
first_residue_numberAF = 1
if first_residue_numberAF != 1:
    print("WARNING: For now this is not implemented. You'll need to hand adjust any values given here.")

Next part of preparation steps though is to get some of the supporting data in light of what has been enterd.

This next cell will get the PDB file specified by the PDB id.

In [5]:
#get stucture
import os
structure_id = structure.upper()
structure_pdb_fn = f"{structure_id}.pdb"
if not os.path.isfile(structure_pdb_fn):
    !curl -OL https://files.rcsb.org/download/{structure_pdb_fn}.gz
    !gunzip {structure_id}.pdb.gz
structures = [structure_id,'AFpred']

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  110k    0  110k    0     0   113k      0 --:--:-- --:--:-- --:--:--  114k


This next cell will get the PDBsum record of the sequence seen.

In [6]:
#The page I set things up to get the PDBsum sequence actually gets HTML and so wrapping it in beautiful soup/
# piping the data into there removes the HTML cruft to just leave the sequence content
# in FASTA format
fasta_seq_file_name = 'the_pdbsum_seq.fa'
from sh import curl
from bs4 import BeautifulSoup
fasta_url = "https://www.ebi.ac.uk/thornton-srv/databases/cgi-bin/pdbsum/GetText.pl"
output_str = curl("-L",
    "--data", "pdb={}&chain={}&seq_fasta={}".format(structure, chain,pdbsum_num),
    fasta_url)
soup = BeautifulSoup(output_str, 'html.parser')
with open(fasta_seq_file_name, 'w') as f:
    f.write(soup.get_text().strip())

That should conclude the preparation steps.

Now we can get into the main processing steps.

-----------------------------------

### Main Processing Steps

This will consist of three main parts:
- Determine the missing residues from the experimental structure and make segments that don't include those. This way the fit commands won't involve anything not present in both the experimental and AlphaFold-predicted structure.
- determine the N-terminal correction value. In other words, if the sequences aren't already corresponding in numbering, what value is needed to correct the solved structure sequence relative the sequence used in AlphaFold. This will be an integer and can be positive or negative. You can always set this by hand yourself if you know. The idea is that this will allow the fit commands to match the corresponding amino acid and target those for the superposition step.
- Synthesize that information to make the molecular visualization fit commands for use with both PyMOL and Jmol/JSmol.

Determine the missing residues from the involved chain and make continguous segments that skip over any missing residue(s).

In [7]:
# Get a list of resolved residue positions for making into ranges that can be used in molecular visualization commands
# relies on use of `not residue.id[0].startswith('H_')` to exclude heteroatoms. (see under 'What is a residue id?' 
# at https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ).
# For example, without use of `not residue.id[0].startswith('H_')` to exclude heteroatoms
# here a heteroatom zinc would be included as part of chain K and numbered as 201 and 
# that would cause issues with the range numbering listing a lone residue that isn't
# actually part of chain K.
from Bio.PDB import *
from collections import defaultdict
structure_obj = PDBParser().get_structure(structure_id, structure_pdb_fn)
residues_resolved_per_chain = defaultdict(list)
for model in structure_obj:
    for chain_id in model:
        for residue in chain_id:
            #print (str(chain.id),residue.id[1])
            #print (str(chain.id),residue.id)
            '''
            if str(chain.id) == 'K':
                print (residue.id[0])
                '''
            #print [residue.id[0]]
            if not residue.id[0].startswith('H_'):
                residues_resolved_per_chain[str(chain_id.id)].append(residue.id[1])
print ('List of resolved residues from first chain:')
print (residues_resolved_per_chain[list(residues_resolved_per_chain.keys())[0]]) #to get first with Python 3 without iter
#print ('chains:',residues_resolved_per_chain.keys()) 
# SUMMARIZE RESOLVED RESIDUES
print ("\nTotal resolved per chain:")
for chain in residues_resolved_per_chain:
    print (chain, len(residues_resolved_per_chain[chain]))

List of resolved residues from first chain:
[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214,

/srv/conda/envs/notebook/lib/python3.10/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5790.
  warnings.warn(


In [8]:
# Get ranges of resolved residue positions that can be used in molecular visualization commands
from Bio.PDB import *
from collections import defaultdict

def range_extract(lst):
    'Yield 2-tuple ranges or 1-tuple single elements from list of increasing ints'
    'modified from  https://www.rosettacode.org/wiki/Range_extraction#Python'
    lenlst = len(lst)
    i = 0
    while i< lenlst:
        low = lst[i]
        while i <lenlst-1 and lst[i]+1 == lst[i+1]: i +=1
        hi = lst[i]
        if hi - low >= 1:     #<---MAIN DIFFERENCE FROM SOURCE
            yield (low, hi)
        else:
            yield (low,)
        i += 1
def printr(ranges):
    # from https://www.rosettacode.org/wiki/Range_extraction#Python
    print( ','.join( (('%i-%i' % r) if len(r) == 2 else '%i' % r)
                     for r in ranges ) )

    
def get_nice_ranges(ranges):
    return [ (('%i-%i' % r) if len(r) == 2 else '%i' % r) for r in ranges ]
    
structure_obj = PDBParser().get_structure(structure_id, structure_pdb_fn)
residues_resolved_per_chain = defaultdict(list)
for model in structure_obj:
    for chain_id in model:
        for residue in chain_id:
            #print (str(chain.id),residue.id[1])
            #print (str(chain.id),residue.id)
            '''
            if str(chain.id) == 'K':
                print (residue.id[0])
                '''
            #print [residue.id[0]]
            if not residue.id[0].startswith('H_'):
                residues_resolved_per_chain[str(chain_id.id)].append(residue.id[1])

#print (residues_resolved_per_chain[list(residues_resolved_per_chain.keys())[0]]) #to get first with Python 3 without iter
#print ('chains:',residues_resolved_per_chain.keys()) 

print ("Residues resolved per chain, concisely stated:")
ranges_o_residues_resolved_per_chain = {}
for k,lst in residues_resolved_per_chain.items():
    print(k)
    #print (list(range_extract(lst))) # to see the list of tuples unformatted
    printr(range_extract(lst)) # tuples formatted as ranges
    ranges_o_residues_resolved_per_chain[k] = get_nice_ranges(range_extract(lst))
first_residue_numberEXP = residues_resolved_per_chain[chain][0]
print(f"{first_residue_numberEXP=}")
ranges_o_residues_resolved_per_chain

Residues resolved per chain, concisely stated:
A
3-614
B
3-50
first_residue_numberEXP=3


/srv/conda/envs/notebook/lib/python3.10/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5790.
  warnings.warn(


{'A': ['3-614'], 'B': ['3-50']}

Next, we can let an alignment determine the N-terminal correction value.  
The idea is that at least at this point we'll know we have the number of the corresponding amino acids and if the submitted sequence was contiguous from that point, the numbering for the fit commands should end up targeting the corresponding amino acids beyond the N-terminus of each of the two involved sequences.  
If you already know this value, just assign `Nterminal_correction_value` as the integer. Leave it as it is, assigned to `None`, if you want the code to determine this.

In [9]:
Nterminal_correction_value = None
# Next, determine the N-terminal correction value. You can set this yourself on the line above to skip this step.

In [10]:
# set the value on the above line if you know it already (or this cell fails to run for whatever reason), otherwise this 
# cell should determine it for you.
#--------------------------------------------------------------------------------------------#
# bulk of this code adapted from `roughly_score_relationships_to_subject_seq_pairwise_premsa.py`
import sys
import os
from Bio.Seq import Seq 
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio import Align
from Bio import pairwise2
#from Bio.SubsMat import MatrixInfo as matlist #seems needed for `pairwise2` that isn't being used here and is indeed 
# deprecated now, because if I try to run that line I get `BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.`

default_block_length = 9141 # I found that in Jupyter sessions hosted on Binder
# that memory was an issue that causes things to silently fail if I tried using
# sequences 9141 bp or longer. FEEL FREE TO ADJUST THIS IF YOU ARE ON A LARGER 
# MACHINE. There are ways to change it for specific jobs on the command line, or 
# when you call the main function. Make this value tiny if things seem to hang for 
# more than five minutes so you can verify everything else is fine.
block_len = default_block_length

###---------------------------HELPER FUNCTIONS---------------------------------###


###--------------------------END OF HELPER FUNCTIONS---------------------------###
###--------------------------END OF HELPER FUNCTIONS---------------------------###


###---------------------------WORK HORSE CLASS-------------------------------###


# this comes straight from https://github.com/berrisfordjohn/adding_stats_to_mmcif/blob/master/adding_stats_to_mmcif/pairwise_align.py

class SequenceAlign:
    # this comes straight from 
    # https://github.com/berrisfordjohn/adding_stats_to_mmcif/blob/master/adding_stats_to_mmcif/pairwise_align.py

    def __init__(self, sequence1, sequence2):
        self.sequence1 = sequence1
        self.sequence2 = sequence2
        self.score = None
        #logging.debug(self.sequence1)
        #logging.debug(self.sequence2)

    def rna(self, seq):
        return set(seq).issubset(set("AUGC"))

    def dna(self, seq):
        return set(seq).issubset(set("ATGC"))

    # def dnarna(self, seq):
    #    return self.rna(seq=seq) or self.dna(seq=seq)

    def both_sequences_same_type(self):
        if self.dna(self.sequence1) == self.dna(self.sequence2) and self.rna(self.sequence1) == self.rna(
                self.sequence2):
            return True, ''
        return False, 'sequences not the same type'
        # if self.dnarna(self.sequence1) != self.dnarna(self.sequence2):
        #    return False, 'sequences not the same type'
        # return True, ''

    def remove_gaps(self, sequence):
        return str(sequence).replace("\n", "").replace(" ", "")

    def prepare_sequences(self):
        self.sequence1 = self.remove_gaps(self.sequence1)
        self.sequence2 = self.remove_gaps(self.sequence2)
        # if len(self.sequence1) > 2000 or len(self.sequence2) > 2000:
        #    return False, 'sequences too long. Please install emboss needle'
        return self.both_sequences_same_type()


    def pairwise_aligner(self):

        aligner = Align.PairwiseAligner()
        aligner.open_gap_score = -10
        aligner.extend_gap_score = -0.5
        aligner.target_end_gap_score = 0.0
        aligner.query_end_gap_score = 0.0
        # aligner.match = 2
        # aligner.mismatch = -1
        # only need to run aligner.score. This improves memory usage and speed. 
        # alignments = aligner.align(self.sequence1, self.sequence2)
        # for alignment in sorted(alignments):
        #    logging.debug(alignment)
        align_score = aligner.score(self.sequence1, self.sequence2)
        #logging.info(align_score)

        self.score = align_score
            # Generate the actual alignments
        alignments = aligner.align(self.sequence1, self.sequence2)
        
        # Store the best alignment
        self.best_alignment = alignments[0] if len(alignments) > 0 else None
        self.score = aligner.score(self.sequence1, self.sequence2)

    """
    def do_alignment_emboss(self):
        needle_cline.asequence = "asis:" + self.sequence1
        needle_cline.bsequence = "asis:" + self.sequence2
        stdout, stderr = needle_cline()
        result = [AlignIO.read(StringIO.StringIO(stdout), "emboss")]
        self.score = self.get_emboss_score(result[0].seq, result[1].seq)
    def get_emboss_score(self, seq1, seq2):
        return sum(aa1 == aa2 for aa1, aa2 in zip(seq1, seq2))
    """

    def get_alignment_score(self):
        return self.score

    def do_sequences_align(self):
        if self.score > 0:
            return True
        return False

    def do_sequence_alignment(self):
        sequences_ok, error = self.prepare_sequences()
        if not sequences_ok:
            return False, error, 0, None
        self.pairwise_aligner()
        if self.do_sequences_align():
            return True, '', self.score, self.best_alignment
        return False, 'sequences do not align', 0, None
###--------------------------END OF WORK HORSE CLASS-------------------------###
###--------------------------END OF WORK HORSE CLASS-------------------------###

#*******************************************************************************
###------------------------'main' function of script---------------------------##

def roughly_score_relationships_to_subject_seq_pairwise_premsa(
    seqs, block_len=default_block_length, return_df = True, save_text_of_df=True):
    '''
    Main function of script. 
    sequences scored of how similar the sequences are to first one in provided
    sequence file.
    based on using 
    https://github.com/berrisfordjohn/adding_stats_to_mmcif/blob/master/adding_stats_to_mmcif/pairwise_align.py

    Optionally also returns a dataframe of the results data, meant for use in a
    Jupyter notebook.
    '''
    # Read in the sequences
    records = []
    for record in SeqIO.parse(seqs, "fasta"):
        records.append(record)
    # feedback
    sys.stderr.write("Sequences read in...")
    max_seq_size = max([len(x.seq) for x in records])
    sys.stderr.write("\nLongest sequence in input detected as "
        "{}.\n...".format(max_seq_size))


    # Prepare to pairwise comparisons to the first one in the list of supplied sequences
    sys.stderr.write("calculating scores of pairwise alignments...")
    subj = records[0]
    sequences_to_compare = records[1:]

    # Limit to handle if exceeds memory limits for aligning
    if max_seq_size > block_len:
        subj = subj[:block_len]
        sequences_to_compare = [x[:block_len] for x in sequences_to_compare]
        sys.stderr.write("\nAnticipated memory issues with long sequence and\n"
            "so only block of {} bps from the start "
            "compared.\n...".format(len(sequences_to_compare[0])))
    
    # clearing records help any on performance? Answer seemed to be 'no'.
    #records = [] # didn't seem to hep as still failed at same point in test
    # with or without that step. Plus this will mean cannot add in another
    # block of aligned sequence for assessment below.


    score_results = {}
    for sequence in sequences_to_compare:
        sa = SequenceAlign(sequence1=subj.seq, sequence2=sequence.seq)
        aligned, error, score = sa.do_sequence_alignment()
        score_results[sequence.id] =  score


    # for those longer than 2x the size of the block the memory resources seem
    # to handle, also assess a block of sequence of the safe length back from 
    # the end to get an even larger representation of the diversity of the 
    # sequence (not going to divide up and assess further because beyond ends
    # might not match up well without alignment anyway and so no point. THIS IS
    # WHOL ENDEAVOR IS MEANT AS QUCIK AND DIRTY ASESSMENT prior to real 
    # alignment of at least some of the sequences and SO NO NEED TO BE
    # THOROUGH.)
    if max_seq_size > (block_len*2):
        subj = records[0]
        sequences_to_compare = records[1:]
        subj = subj[-block_len:]
        sequences_to_compare = [x[-block_len:] for x in sequences_to_compare]
        sys.stderr.write("\nSuper long sequence detected: Comparing a 2nd "
            "block back from the\nsequence 'end' as well and "
            "combining scores.\n...")
        for sequence in sequences_to_compare:
            sa = SequenceAlign(sequence1=subj.seq, sequence2=sequence.seq)
            aligned, error, score = sa.do_sequence_alignment()
            score_results[sequence.id] +=  score


    # Make dataframe of the results and sort on score
    sys.stderr.write("summarizing scores...")
    import pandas as pd
    df = pd.DataFrame(list(score_results.items()), columns = ['id', 'score_vs_'+subj.id])
    df = df.sort_values('score_vs_'+subj.id, ascending=False)
        

    # feedback
    sys.stderr.write("\nResults converted to a dataframe...")


    # Reporting and Saving
    #---------------------------------------------------------------------------
    #print(df)#originally for debugging during development,added..
    # Document the full set of data collected in the terminal or 
    # Jupyter notebook display in some manner. 
    # Using `df.to_string()` because more universal than `print(df)` 
    # or Jupyter's `display(df)`.
    #sys.stderr.write("\nFor documenting purposes, the following lists the "
    #    "parsed data:\n")
    #with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    #    display(df)
    #sys.stderr.write(df.to_string())

    # Handle saving the dataframe
    if save_text_of_df == False:
        sys.stderr.write("\n\nTabular text of the data "
        "was not stored for use\nelsewhere "
        "because `no_table` was specified.")
    else:
        df.to_csv(df_save_as_name, sep='\t',index = False)
        # Let user know
        sys.stderr.write("\n\nA table of the data "
        "has been saved as a text file (tab-delimited).\n"
        "DATA is stored as ==> '{}'".format(df_save_as_name ))

    
    # Return dataframe (optional)
    #---------------------------------------------------------------------------
    if return_df:
        sys.stderr.write("\n\nReturning a dataframe with the information "
                "as well.")
        return df

###--------------------------END OF MAIN FUNCTION----------------------------###
###--------------------------END OF MAIN FUNCTION----------------------------###

# Read in the sequence present in the experimental structure
records = []
for record in SeqIO.parse(fasta_seq_file_name, "fasta"):
    records.append(record)
# feedback
sys.stderr.write("Sequences read in...")
max_seq_size = max([len(x.seq) for x in records])
sys.stderr.write("\nLongest sequence in input detected as "
    "{}.\n...".format(max_seq_size))

# Make a BioPython seq record out of the the AlphaFold-submitted sequence
submitted_record = SeqRecord(
    Seq(submitted_seq),
    id="submitted_seq",
    description="AlphaFold submitted sequence"
)

# Perform pairwise comparison  to the the experimental structure with the AlphaFold-submitted sequence
sys.stderr.write("calculating pairwise alignment...")
subj = records[0]


sa = SequenceAlign(sequence1=subj.seq, sequence2=submitted_record.seq)
aligned, error, score, alignment = sa.do_sequence_alignment()


if not Nterminal_correction_value:
    print("Preparing to determine offset value using an alignment...")
    print("Preparing alignment of the experimental and predicted sequence...")

if aligned:
    print(alignment)
    print(f"Score: {score}")
    
    # Get the start positions for target and query
    target_start = alignment.aligned[0][0][0]  # Start position in target sequence
    query_start = alignment.aligned[1][0][0]   # Start position in query sequence
    
    print(f"Target starts at position: {target_start}")
    print(f"Query starts at position: {query_start}")
    print(f"Offset for target: {target_start}")
    print(f"Offset for query: {query_start}")
    print(f"First residue number authors assign to first resolved residue: {first_residue_numberEXP}")
    Nterminal_correction_value = int((query_start - first_residue_numberEXP) + 1)
    print(f"{Nterminal_correction_value=}")

Preparing to determine offset value using an alignment...
Preparing alignment of the experimental and predicted sequence...
target            0 --------------------------EKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEV
                  0 --------------------------||||||||||||||||||||||||||||||||||
query             0 MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEV

target           34 VLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALA
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 VLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALA

target           94 VLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDF
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           120 VLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDF

target          154 QNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGC
                180 |||||||||||||||||||||||||||||||||||||||||||

/srv/conda/envs/notebook/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
Sequences read in...
Longest sequence in input detected as 612.
...calculating pairwise alignment.../srv/conda/envs/notebook/lib/python3.10/site-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'target_end_gap_score' was renamed to 'end_insertion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.10/site-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_end_gap_score' was renamed to 'end_deletion_score'. This was do

Now with these details in hand, the following code can run in order to calculate the fit commands.

In [11]:
# much of this part is adapted from `Using Biopython PDB module to list resolved residues and construct fit commands.ipynb` and `Determine residues that match to a reference from MSA and use to construct fit commands.ipynb
chain_pair = [('A','A')] #putting chain identiifier from 'experimenta' structure first in each pair.

#define `ranges_o_residues_resolved_per_chain_pair` for the pair
shared = list(set(residues_resolved_per_chain[chain_pair[0][0]]) )
shared_positions_per_chain_pair = {chain_pair[0][0]: shared}
#print(shared_positions_per_chain_pair)
ranges_o_residues_resolved_per_chain_pair = {chain_pair[0][0]: ranges_o_residues_resolved_per_chain[chain_pair[0][0]]}
ranges_o_residues_resolved_per_chain_pair
print(ranges_o_residues_resolved_per_chain_pair)


def adjust_for_correction_val(list_of_range_txt):
    '''
    adjust range using Nterminal_correction_value
    '''
    corrected_range_list = []
    for x in list_of_range_txt:
        the_two_numbers = x.split('-',1)
        fixed_first = int(the_two_numbers[0]) + Nterminal_correction_value
        fixed_second = int(the_two_numbers[1]) + Nterminal_correction_value
        corrected_range_list.append(f'{fixed_first}-{fixed_second}')
    return corrected_range_list

         
# form commands
formatted_commands = ""
selection_pairs = [] #to collect two item tuple elements, each item being a name
for chain_pair_id in ranges_o_residues_resolved_per_chain_pair:
    print(chain_pair_id)
    selection_name_pair = []
    print(chain_pair[0])
    for indx,chain_id in enumerate(chain_pair[0]):
        #print(f"{indx=}")
        #print(f"{chain_id=}")
        if indx == 0:
            print(ranges_o_residues_resolved_per_chain_pair[chain_pair_id])
            sel_nom = f"{structures[indx]}ch{chain_id}CA" #`CA` at end stands for `name CA` / alpha-carbon
            formatted_commands += f"select {sel_nom},"
            formatted_commands += "|".join( f" {structures[indx]} and resid {pos_range} and chain {chain_id} and name CA " 
                                           for pos_range in ranges_o_residues_resolved_per_chain_pair[chain_pair_id] ) 
            #for pos_range in ranges_o_residues_resolved_per_chain_pairs[chain_pair]:
             #   formatted_commands += f"{structures[indx].id} and resid {pos_range} and chain {chain} and name CA"
            formatted_commands += "\n"
            selection_name_pair.append(sel_nom)
        else:
            AFranges_o_residues_resolved_per_chain_pair = ranges_o_residues_resolved_per_chain_pair.copy()
            AFranges_o_residues_resolved_per_chain_pair[chain_pair_id] = adjust_for_correction_val(ranges_o_residues_resolved_per_chain_pair[chain_pair_id])
            sel_nom = f"{structures[indx]}ch{chain_id}CA" #`CA` at end stands for `name CA` / alpha-carbon
            formatted_commands += f"select {sel_nom},"
            formatted_commands += "|".join( f" {structures[indx]} and resid {pos_range} and chain {chain_id} and name CA " 
                                           for pos_range in AFranges_o_residues_resolved_per_chain_pair[chain_pair_id] ) 
            #for pos_range in ranges_o_residues_resolved_per_chain_pairs[chain_pair]:
             #   formatted_commands += f"{structures[indx].id} and resid {pos_range} and chain {chain} and name CA"
            formatted_commands += "\n"
            selection_name_pair.append(sel_nom)
    selection_pairs.append(selection_name_pair)
print(selection_pairs)
first_structure_selections = "|".join(f" {x[0]} " for x in selection_pairs)
second_structure_selections = "|".join(f" {x[1]} " for x in selection_pairs)
formatted_commands += f"pair_fit ( {first_structure_selections}), ( {second_structure_selections})"

# TO DO: JMOL / JSMOL COMMANDS
# see `Determine residues that match to a reference from MSA and use to construct fit commands.ipynb` because Jmol part was done there

#shared_positions_per_chain_pairs
#ranges_o_residues_resolved_per_chain_pairs
print("\n\n")
print ("FORMATTED PYMOL COMMANDS:")
print(" ")
print(formatted_commands)

{'A': ['3-614']}
A
('A', 'A')
['3-614']
[['1NQLchACA', 'AFpredchACA']]



FORMATTED PYMOL COMMANDS:
 
select 1NQLchACA, 1NQL and resid 3-614 and chain A and name CA 
select AFpredchACA, AFpred and resid 27-638 and chain A and name CA 
pair_fit (  1NQLchACA ), (  AFpredchACA )


Next, change the values above and run it for your favorite structure and prediction.

------

Enjoy!